# Dataset Processing and Evaluation

In [3]:
import pandas as pd
import numpy as np
from ast import literal_eval
import os
import glob

In [4]:
# PATH = '/data/puccetti/space_data/all_run_tiny.csv'
current_dir = os.getcwd()
project_name = "Zero-Trust-Authentication-Based-on-ROS2"

if project_name in current_dir:
    root_dir = current_dir[:current_dir.find(project_name) + len(project_name)]
else:
    root_dir = current_dir

# 設定目標資料夾
input_dir = os.path.join(root_dir, "rospace_dataset", "1_processing", "merged-dataset")
output_dir = os.path.join(root_dir, "rospace_dataset", "3_complete_dataset")
os.makedirs(output_dir, exist_ok=True) 

# 自動抓取最新且非 unlabelled 的 csv
search_pattern = os.path.join(input_dir, "merged-*.csv")
valid_files = [f for f in glob.glob(search_pattern) if "unlabelled" not in f]

if not valid_files:
    raise FileNotFoundError(f"找不到已標註的 merged-*.csv 檔案於 {input_dir}")

# 挑選修改時間最新的檔案
latest_file = max(valid_files, key=os.path.getmtime)
PATH = latest_file
print(f"自動鎖定檔案: {PATH}")

自動鎖定檔案: c:\Users\chuni\Desktop\Zero-Trust-Authentication-Based-on-ROS2\rospace_dataset\1_processing\merged-dataset\merged-0531.csv


In [5]:
pd.set_option("display.max_columns", None)
pd.get_option("display.max_columns")

In [6]:
df = pd.read_csv(PATH, low_memory=False)

## Some checks on columns and values

Check unique values in the label ('attack' column)

In [7]:
print(df.shape)
print(df['attack'].value_counts())

(78174, 806)
attack
observe                 58257
nmap port scanning      14659
nmap SYN flood           2577
metasploit SYN flood     2497
ros2 reconnaissance       136
nmap discovery             40
ros2 node crashing          6
ros2 reflection             2
Name: count, dtype: int64


Print the size of the dataset

In [8]:
print(df.shape)

(78174, 806)


Delete some unuseful columns: 
- 'Unnamed' columns are just duplicate indexes of dataframes

In [9]:
subs = "Unnamed"
res = [i for i in df.columns if subs in i]
print(len(res))
print(res)
df=df.drop(res, axis=1)

1
['Unnamed: 0']


Search for "duplicate" substring in all columns as contains target ip 

In [10]:
subs = "Duplicate"
res = [i for i in df.columns if subs in i]
print(res)

[]


In [11]:
df = df.drop(res, axis=1)
print(df.shape)

(78174, 805)


In [12]:
print(df.columns)

Index(['timestamp', 'layers.frame.frame.interface_id',
       'layers.frame.frame.interface_id_tree.frame.interface_name',
       'layers.frame.frame.encap_type', 'layers.frame.frame.time',
       'layers.frame.frame.time_utc', 'layers.frame.frame.offset_shift',
       'layers.frame.frame.time_relative', 'layers.frame.frame.number',
       'layers.frame.frame.len',
       ...
       'Tcp_Close', 'nr_active_file', 'nr_inactive_file', 'topic_name',
       'src_topic', 'subscribers_count', 'publisher_count', 'msg_type',
       'msg_data', 'attack'],
      dtype='str', length=805)


## Delete columns with 0 or 1  unique values

In [13]:
# for col in df.columns:
#     n_unique = len(df[col].unique())
#     if n_unique == 1 or n_unique == 0:
#         df.drop(col,inplace=True, axis=1)
print("正在掃描並清除數值毫無變化的欄位...")
drop_cols = []
protected_cols = ['attack', 'timestamp'] # 絕對不能刪除的特徵名單

for col in df.columns:
    if col in protected_cols:
        continue # 如果是受保護的欄位，直接跳過不檢查

    n_unique = df[col].nunique(dropna=False)
    if n_unique <= 1:
        drop_cols.append(col)

df.drop(drop_cols, axis=1, inplace=True)
print(f"🧹 已清除 {len(drop_cols)} 個零變異欄位")

正在掃描並清除數值毫無變化的欄位...
🧹 已清除 119 個零變異欄位


In [14]:
print(df.shape)

(78174, 686)


In [15]:
# df=df.drop(['expNumber', '#'], axis=1)
df=df.drop(['expNumber', '#'], axis=1, errors='ignore')

In [16]:
print(df.shape)

(78174, 686)


In [17]:
# np.save('/data/puccetti/space_data/all_features.npy', df.columns, allow_pickle=True)
filename = os.path.basename(latest_file)
date_val = filename.replace("merged-", "").replace(".csv", "")
save_path = os.path.join(output_dir, f'all_features_{date_val}.npy')

np.save(save_path, df.columns, allow_pickle=True)
print(f"特徵名單已成功存至: {save_path}")

# (強烈建議保留此行) 將清洗完的乾淨資料存成 CSV，以便後續訓練模型使用
clean_csv_target = os.path.join(output_dir, f'cleaned_merged-{date_val}.csv')
df.to_csv(clean_csv_target, index=False)

特徵名單已成功存至: c:\Users\chuni\Desktop\Zero-Trust-Authentication-Based-on-ROS2\rospace_dataset\3_complete_dataset\all_features_0531.npy


In [18]:
import pandas as pd
import numpy as np
import os
import glob

# 1. 自動定位專案根目錄
current_dir = os.getcwd()
project_name = "Zero-Trust-Authentication-Based-on-ROS2"
if project_name in current_dir:
    root_dir = current_dir[:current_dir.find(project_name) + len(project_name)]
else:
    root_dir = current_dir

output_dir = os.path.join(root_dir, "rospace_dataset", "3_complete_dataset")

# 2. 🕵️ 自動抓取最新的 cleaned_merged 檔案
search_pattern = os.path.join(output_dir, "cleaned_merged-*.csv")
csv_files = glob.glob(search_pattern)

if not csv_files:
    raise FileNotFoundError(f"❌ 在 {output_dir} 找不到任何 cleaned_merged 檔案！請確認你已經跑過清洗程式。")

# 挑選最新修改的那個 CSV
latest_csv = max(csv_files, key=os.path.getmtime)
csv_path = latest_csv

# 自動從檔名把 date_val (代號) 抽出來，用來找失散的雙胞胎 .npy
filename = os.path.basename(latest_csv)
date_val = filename.replace("cleaned_merged-", "").replace(".csv", "")
npy_path = os.path.join(output_dir, f'all_features_{date_val}.npy')

print(f"🤖 自動鎖定最新檔案：{filename} (代號: {date_val})")
print("-" * 40)
print("🔍 開始執行資料健康度檢查...\n")

try:
    # 讀取剛剛產出的兩個檔案
    df = pd.read_csv(csv_path, low_memory=False)
    saved_features = np.load(npy_path, allow_pickle=True)

    # --- 檢查 1：檔案維度與對齊度 ---
    print("✅ [檢查 1] 檔案維度與對齊度")
    print(f"  - CSV 資料筆數 (Rows): {df.shape[0]} 筆")
    print(f"  - CSV 特徵數量 (Columns): {df.shape[1]} 個")
    print(f"  - NPY 特徵數量: {len(saved_features)} 個")
    if df.shape[1] == len(saved_features):
        print("  🟢 狀態: 完美！CSV 與 NPY 的特徵數量完全一致。")
    else:
        print("  🔴 警告: 數量不一致，請檢查存檔過程。")

    # --- 檢查 2：垃圾欄位是否真的清除了 ---
    print("\n✅ [檢查 2] 垃圾欄位清除確認")
    unnamed_cols = [c for c in df.columns if 'Unnamed' in c]
    duplicate_cols = [c for c in df.columns if 'Duplicate' in c]
    if len(unnamed_cols) == 0 and len(duplicate_cols) == 0:
        print("  🟢 狀態: 完美！沒有殘留的 Unnamed 或 Duplicate 欄位。")
    else:
        print(f"  🔴 警告: 發現殘留！Unnamed: {len(unnamed_cols)}個, Duplicate: {len(duplicate_cols)}個")

    # --- 檢查 3：死水欄位 (零變異) 是否還在 ---
    print("\n✅ [檢查 3] 零變異欄位確認")
    zero_var_count = sum(df.nunique(dropna=False) <= 1)
    if zero_var_count == 0:
        print("  🟢 狀態: 完美！所有留下來的特徵都有數值變化，對 AI 有幫助。")
    else:
        print(f"  🔴 警告: 還有 {zero_var_count} 個從頭到尾數值都沒變的廢物欄位。")

    # --- 檢查 4：最重要的「攻擊標籤 (Label)」是否正常 ---
    print("\n✅ [檢查 4] 攻擊標籤 (attack) 分佈狀況")
    if 'attack' in df.columns:
        print(df['attack'].value_counts(dropna=False))
        if df['attack'].isnull().sum() > 0:
            print("  ⚠️ 提醒: 你的標籤中有 NaN (空值)，可能代表某些時間點沒有攻擊紀錄，訓練時可能需要濾掉或填補。")
    else:
        print("  🔴 致命錯誤: 'attack' 欄位不見了！模型會沒有目標可以訓練！")

except FileNotFoundError as e:
    print(f"❌ 找不到檔案: {e}")

🤖 自動鎖定最新檔案：cleaned_merged-0531.csv (代號: 0531)
----------------------------------------
🔍 開始執行資料健康度檢查...

✅ [檢查 1] 檔案維度與對齊度
  - CSV 資料筆數 (Rows): 78174 筆
  - CSV 特徵數量 (Columns): 686 個
  - NPY 特徵數量: 686 個
  🟢 狀態: 完美！CSV 與 NPY 的特徵數量完全一致。

✅ [檢查 2] 垃圾欄位清除確認
  🟢 狀態: 完美！沒有殘留的 Unnamed 或 Duplicate 欄位。

✅ [檢查 3] 零變異欄位確認
  🟢 狀態: 完美！所有留下來的特徵都有數值變化，對 AI 有幫助。

✅ [檢查 4] 攻擊標籤 (attack) 分佈狀況
attack
observe                 58257
nmap port scanning      14659
nmap SYN flood           2577
metasploit SYN flood     2497
ros2 reconnaissance       136
nmap discovery             40
ros2 node crashing          6
ros2 reflection             2
Name: count, dtype: int64
